# **Week 4 Assignment: Saliency Maps (PyTorch)**

Welcome to the final programming exercise of this course! For this week, your task is to adapt the Cats vs Dogs Class Activation Map ungraded lab (the second ungraded lab of this week) and make it generate saliency maps instead.

As discussed in the lectures, a saliency map shows the pixels which greatly impacts the classification of an image.
- This is done by getting the gradient of the loss with respect to changes in the pixel values, then plotting the results.
- From there, you can see if your model is looking at the correct features when classifying an image.
  - For example, if you're building a dog breed classifier, you should be wary if your saliency map shows strong pixels outside the dog itself (e.g. sky, grass, dog house, etc...).

In this assignment you will be given prompts but less starter code to fill in in.
- It's good practice for you to try and write as much of this code as you can from memory and from searching the web.
- **Whenever you feel stuck**, please refer back to the labs of this week to see how to write the code. In particular, look at:
  - **Ungraded Lab 2: Cats vs Dogs CAM**
  - **Ungraded Lab 3: Saliency**

> This is a PyTorch port of the original TensorFlow assignment. The exercises are the same, but you will write PyTorch code: a `Dataset`/`DataLoader` pipeline, an `nn.Sequential` model, an explicit training loop, and `torch.autograd` for the gradients of the saliency map. The pre-trained weights are provided as Keras `.h5` files; a helper is included to copy them into your PyTorch model, so the model must follow the expected architecture exactly. Note that the Coursera autograder expects the TensorFlow version, so the PyTorch results cannot be submitted for grading.

### Download test files and weights

Let's begin by first downloading files we will be using for this lab.

In [ ]:
import os
import urllib.request

files_to_download = {
    # the same test files from the Cats vs Dogs ungraded lab
    'cat1.jpg': 'https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/cat1.jpeg',
    'cat2.jpg': 'https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/cat2.jpeg',
    'catanddog.jpg': 'https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/catanddog.jpeg',
    'dog1.jpg': 'https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/dog1.jpeg',
    'dog2.jpg': 'https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/dog2.jpeg',
    # prepared weights (Keras .h5 format)
    '0_epochs.h5': 'https://docs.google.com/uc?export=download&id=1kipXTxesGJKGY1B8uSPRvxROgOH90fih',
    '15_epochs.h5': 'https://docs.google.com/uc?export=download&id=1oiV6tjy5k7h9OHGTQaf0Ohn3FmF-uOs1',
}
for name, url in files_to_download.items():
    if not os.path.exists(name):
        urllib.request.urlretrieve(url, name)
print("files ready")

### Import the required packages

Please import:

  * PyTorch (`torch`, `torch.nn`, `torch.nn.functional`)
  * `Dataset` and `DataLoader` from `torch.utils.data`
  * torchvision `transforms`
  * Numpy
  * Matplotlib's PyPlot
  * OpenCV (cv2)
  * PIL's `Image` (to save the saliency maps)

In [ ]:
# PyTorch and its neural network / functional / data helpers
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# torchvision preprocessing
from torchvision import transforms

# numerics, plotting, image IO
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

### Download and prepare the dataset.

#### Load Cats vs Dogs

The dataset is the Microsoft/Kaggle *Cats vs Dogs* archive (the same images that TensorFlow Datasets serves as `cats_vs_dogs`). The two cells below download it and define a small `Dataset` class for you, exactly as in the Cats vs Dogs CAM lab.

In [ ]:
import os
import urllib.request
import zipfile

data_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
data_file_name = "data/kagglecatsanddogs_5340.zip"
os.makedirs("data", exist_ok=True)

if not os.path.exists(data_file_name):
    print("downloading the dataset (about 800 MB)...")
    urllib.request.urlretrieve(data_url, data_file_name)
if not os.path.exists("data/catsdogs/PetImages"):
    with zipfile.ZipFile(data_file_name, 'r') as zip_ref:
        zip_ref.extractall("data/catsdogs/")
print("dataset ready")

In [ ]:
import platform
from PIL import Image, ImageFile

# DataLoader worker processes on macOS are started with "spawn", which cannot see classes defined
# inside a notebook (such as the Dataset below). "fork" works fine for the image decoding the workers do.
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None

# a few images in this dataset are truncated; let PIL load what it can instead of raising
ImageFile.LOAD_TRUNCATED_IMAGES = True


def list_cats_vs_dogs_files(root="data/catsdogs/PetImages", cache="data/catsdogs/file_list.txt"):
    '''
    Returns a sorted list of (path, label) pairs, label 0 for cats and 1 for dogs.

    Files that cannot be decoded as images are skipped (like tfds does when it prepares the
    dataset). Checking all 25,000 files takes a minute, so the result is cached on disk.

    Args:
      root (string) -- folder holding the Cat and Dog subfolders
      cache (string) -- file the resulting list is cached in

    Returns:
      list -- (path, label) pairs, sorted and stable across runs
    '''
    if os.path.exists(cache):
        with open(cache) as f:
            return [(line.split("\t")[0], int(line.split("\t")[1])) for line in f.read().splitlines()]

    files = []
    for label, folder in enumerate(["Cat", "Dog"]):
        for name in sorted(os.listdir(os.path.join(root, folder))):
            path = os.path.join(root, folder, name)
            try:
                with Image.open(path) as img:
                    img.verify()
                files.append((path, label))
            except Exception:
                pass   # not a valid image
    with open(cache, "w") as f:
        f.write("\n".join(f"{p}\t{l}" for p, l in files))
    return files


def take_split(files, start, end):
    '''
    The equivalent of the tfds split syntax train[start:end], with the bounds as fractions.

    Args:
      files (list) -- the full list of (path, label) pairs
      start (float) -- fraction of the list to start at, 0.0 is the beginning
      end (float) -- fraction of the list to stop at, 1.0 is the end

    Returns:
      list -- the selected slice of (path, label) pairs
    '''
    n = len(files)
    return files[int(start * n):int(end * n)]


class CatsVsDogs(Dataset):
    '''yields (image, label) pairs; `transform` turns the PIL image into a tensor'''

    def __init__(self, files, transform):
        '''
        Stores the (path, label) pairs and the transform applied to each image.

        Args:
          files (list) -- (path, label) pairs, label 0 for cat and 1 for dog
          transform (callable) -- turns a PIL image into a tensor
        '''
        self.files = files
        self.transform = transform

    def __len__(self):
        '''
        Reports how many items this split holds.

        Returns:
          int -- number of images in this split
        '''
        return len(self.files)

    def __getitem__(self, idx):
        '''
        Loads image `idx`, forces it to RGB, and applies the transform.

        Args:
          idx (int) -- index of the image to fetch

        Returns:
          (tensor, int) -- the transformed image and its label
        '''
        path, label = self.files[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label

* Required: Use `list_cats_vs_dogs_files()` and `take_split()` to fetch the file list and create your training set from the first 80% of the images.

* Optional: You can create validation and test sets from the remaining 20% of the images (i.e. you already used 80% for the train set). This is if you intend to train the model beyond what is required for submission.

In [ ]:
# Load the data and create the train set (optional: val and test sets)

files = list_cats_vs_dogs_files()

# the first 80% of the images become the training set, matching train[:80%] in tfds
train_files = take_split(files, 0.0, 0.8)

# optional: carve validation and test sets out of the remaining 20%
validation_files = take_split(files, 0.8, 0.9)
test_files = take_split(files, 0.9, 1.0)

print(f"{len(train_files)} training, {len(validation_files)} validation, {len(test_files)} test images")

#### Create preprocessing function

Define the preprocessing (a `transforms.Compose`, or a function that takes a PIL image). This will:
  * resize the image to 300 x 300
  * convert it to a float32 tensor of shape `(3, 300, 300)`
  * normalize the pixel values to [0, 1]

(`transforms.ToTensor()` does the last two steps in one go.)

In [ ]:
augmentimages = transforms.Compose([
    transforms.Resize((300, 300)),   # resize every image to 300 x 300
    transforms.ToTensor(),           # (H, W, 3) uint8 -> (3, 300, 300) float32 scaled to [0, 1]
])

#### Preprocess the training set

Create a `CatsVsDogs` dataset from your training files and pass in the preprocessing you just defined.

In [ ]:
augmented_training_data = CatsVsDogs(train_files, augmentimages)

#### Create batches of the training set.

This is already provided for you. Normally, you will want to shuffle the training set. But for predictability, we will simply create the batches.

```Python
# Shuffle the data if you're working on your own personal project
train_batches = DataLoader(augmented_training_data, batch_size=32, shuffle=True, num_workers=4, multiprocessing_context=MP_CONTEXT)
```

In [ ]:
train_batches = DataLoader(augmented_training_data, batch_size=32, num_workers=4, multiprocessing_context=MP_CONTEXT, persistent_workers=True)

### Build the Cats vs Dogs classifier

You'll define a model that is nearly the same as the one in the Cats vs. Dogs CAM lab.
* Please preserve the architecture of the model in the Cats vs Dogs CAM lab (this week's second lab) except for the final `Linear` layer.
* You should modify the Cats vs Dogs model at the last linear layer to output 2 neurons instead of 1.
  - This is because you will adapt the `do_salience()` function from the lab and that works with one-hot encoded labels.
  - You can do this by changing the `out_features` argument of the output `Linear` layer from 1 to 2, with one for each of the classes (i.e. cats and dogs).
  - The model should output the raw class scores (logits). You will turn them into probabilities for the 2 classes (i.e. categories) where the sum of the probabilities adds up to 1 with `softmax` when you compute the saliency map, and `nn.CrossEntropyLoss` will take care of it during training.
* Build it with `nn.Sequential`, using `nn.AdaptiveAvgPool2d(1)` + `nn.Flatten()` for the global average pooling, and move it to the `device`.
* The pre-trained weights are loaded by matching your `Conv2d`/`Linear` layers **in order**, so the layer order must match the expected output below.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Same architecture as the Cats vs Dogs CAM lab, except the final layer has 2 outputs
# instead of 1, so it pairs with the one-hot targets used by do_salience below.
# The layers are given in order so load_keras_weights can match them to the .h5 file.
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),

    nn.AdaptiveAvgPool2d(1),   # GlobalAveragePooling2D: (N, 128, 37, 37) -> (N, 128, 1, 1)
    nn.Flatten(),              # shape: -> (N, 128)
    nn.Linear(128, 2),         # 2 class logits; softmax is applied where probabilities are needed
).to(device)

print(model)
print("Total params:", sum(p.numel() for p in model.parameters()))

**Expected Output:**

```txt
Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (10): ReLU()
  (11): AdaptiveAvgPool2d(output_size=1)
  (12): Flatten(start_dim=1, end_dim=-1)
  (13): Linear(in_features=128, out_features=2, bias=True)
)
Total params: 97698
```

### Loading the prepared Keras weights

The weights for this assignment were saved with Keras (`.h5` files). The helper below copies them into your PyTorch model layer by layer:
- Keras stores convolution kernels as `(height, width, in_channels, out_channels)`; PyTorch uses `(out_channels, in_channels, height, width)`.
- Keras stores dense kernels as `(in_features, out_features)`; PyTorch `Linear` uses `(out_features, in_features)`.

This code is provided for you.

In [ ]:
import h5py

def load_keras_weights(model, path):
    '''
    Copies the weights of a Keras .h5 weights file into a PyTorch model with the same
    sequence of convolution / linear layers.

    Args:
      model (nn.Module) -- the PyTorch model
      path (string) -- path to the .h5 file
    '''
    torch_layers = [m for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]

    with h5py.File(path, 'r') as f:
        layer_names = [n.decode() if isinstance(n, bytes) else n for n in f.attrs['layer_names']]
        # keep only the Keras layers that actually have weights (conv2d, dense)
        keras_layers = [n for n in layer_names if len(f[n].attrs.get('weight_names', [])) > 0]
        assert len(keras_layers) == len(torch_layers), \
            f"the model has {len(torch_layers)} weight layers but the file has {len(keras_layers)}"

        for name, layer in zip(keras_layers, torch_layers):
            weight_names = [w.decode() if isinstance(w, bytes) else w for w in f[name].attrs['weight_names']]
            kernel = np.array(f[name][[w for w in weight_names if 'kernel' in w][0]])
            bias = np.array(f[name][[w for w in weight_names if 'bias' in w][0]])
            if isinstance(layer, nn.Conv2d):
                kernel = kernel.transpose(3, 2, 0, 1)   # (kh, kw, in, out) -> (out, in, kh, kw)
            else:
                kernel = kernel.T                       # (in, out) -> (out, in)
            with torch.no_grad():
                layer.weight.copy_(torch.from_numpy(kernel))
                layer.bias.copy_(torch.from_numpy(bias))
    print(f"loaded weights from {path}")

### Create a function to generate the saliency map

Complete the `do_salience()` function below to save the **normalized_tensor** image.
- The major steps are listed as comments below.
  - Each section may involve multiple lines of code.
- Try your best to write the code from memory or by performing web searches.
  - Whenever you get stuck, you can review the "saliency" lab (the third lab of this week) to help remind you of what code to write
- In PyTorch, "watching" a tensor means setting `requires_grad_(True)` on it before the forward pass; `loss.backward()` (or `torch.autograd.grad`) then gives you `image.grad`.

In [ ]:
def do_salience(image, model, label, prefix):
  '''
  Generates the saliency map of a given image.

  Args:
    image (file) -- picture that the model will classify
    model (nn.Module) -- your cats and dogs classifier
    label (int) -- ground truth label of the image
    prefix (string) -- prefix to add to the filename of the saliency map
  '''

  # Read the image and convert channel order from BGR to RGB
  img = cv2.imread(image)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  # Resize the image to 300 x 300 and normalize pixel values to the range [0, 1]
  img = cv2.resize(img, (300, 300)) / 255.0

  # Convert to a float tensor with the channels first, add an additional dimension (for the batch),
  # move it to the device and save this in a new variable
  image_tensor = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0).to(device)   # shape: (1, 3, 300, 300)

  # Declare the number of classes
  num_classes = 2

  # Define the expected output array by one-hot encoding the label
  # The length of the array is equal to the number of classes
  expected_output = F.one_hot(torch.tensor([label]), num_classes).float().to(device)      # shape: (1, 2)

  # Set requires_grad on the float32 image tensor so that gradients are tracked for it
  image_tensor.requires_grad_(True)

  # Put the model in eval mode and get the model's prediction by passing in the image
  # (apply softmax to the logits to get probabilities)
  model.eval()
  predictions = torch.softmax(model(image_tensor), dim=1)
  print(f"predictions: {predictions.detach().cpu().numpy()}  (sums to {float(predictions.sum()):.3f})")

  # Compute an appropriate loss between the expected output and model predictions.
  loss = F.binary_cross_entropy(predictions, expected_output)

  # get the gradients of the loss with respect to the model's input image
  loss.backward()
  gradients = image_tensor.grad                                                            # shape: (1, 3, 300, 300)

  # generate the grayscale tensor (sum the absolute gradients over the channel axis)
  grayscale_tensor = gradients.abs().sum(dim=1)                                            # shape: (1, 300, 300)

  # normalize the pixel values to be in the range [0, 255].
  # the max value in the grayscale tensor will be pushed to 255.
  # the min value will be pushed to 0.
  # Use the formula: 255 * (x - min) / (max - min)
  # Use torch.max, torch.min
  # Cast the tensor as a torch.uint8
  normalized_tensor = (
      255
      * (grayscale_tensor - torch.min(grayscale_tensor))
      / (torch.max(grayscale_tensor) - torch.min(grayscale_tensor))
  ).to(torch.uint8)

  # Remove dimensions that are size 1 and move the tensor to the CPU as a numpy array
  normalized_tensor = normalized_tensor.squeeze().cpu().numpy()                            # shape: (300, 300)

  # plot the normalized tensor
  # Set the figure size to 8 by 8
  # do not display the axis
  # use the 'gray' colormap
  # This code is provided for you.
  plt.figure(figsize=(8, 8))
  plt.axis('off')
  plt.imshow(normalized_tensor, cmap='gray')
  plt.show()

  # optional: superimpose the saliency map with the original image, then display it.
  # we encourage you to do this to visualize your results better
  gradient_color = cv2.applyColorMap(normalized_tensor, cv2.COLORMAP_HOT) / 255.0
  super_imposed = cv2.addWeighted(img, 0.5, gradient_color, 0.5, 0.0)
  plt.figure(figsize=(8, 8))
  plt.axis('off')
  plt.imshow(super_imposed)
  plt.show()

  # save the normalized tensor image to a file. this is already provided for you.
  salient_image_name = prefix + image
  Image.fromarray(normalized_tensor, mode='L').save(salient_image_name, quality=100)

### Generate saliency maps with untrained model

As a sanity check, you will load initialized (i.e. untrained) weights and use the function you just implemented.
- This will check if you built the model correctly and are able to create a saliency map.

If an error pops up when loading the weights or the function does not run, please check your implementation for bugs.
- You can check the ungraded labs of this week.

Please apply your `do_salience()` function on the following image files:

* `cat1.jpg`
* `cat2.jpg`
* `catanddog.jpg`
* `dog1.jpg`
* `dog2.jpg`

Cats will have the label `0` while dogs will have the label `1`.
- For the catanddog, please use `0`.
- For the prefix of the salience images that will be generated, please use the prefix `epoch0_salient`.

In [ ]:
# load initial weights
load_keras_weights(model, '0_epochs.h5')

# generate the saliency maps for the 5 test images
# cats are label 0, dogs are label 1; catanddog uses 0 as the instructions say
test_images = [('cat1.jpg', 0), ('cat2.jpg', 0), ('catanddog.jpg', 0), ('dog1.jpg', 1), ('dog2.jpg', 1)]

for image_name, image_label in test_images:
    do_salience(image_name, model, image_label, 'epoch0_salient')

With untrained weights, you will see something like this in the output.
- You will see strong pixels outside the cat that the model uses that when classifying the image.
- After training that these will slowly start to localize to features inside the pet.

<img src='https://drive.google.com/uc?export=view&id=1h5wP52lwbBUMVLlsgyb-tQl_I9eu42X7' alt='saliency'>

### Configure the model for training

Define the loss function and optimizer.

* Choose a loss function for the model to use when training.
  - The ground truth labels from the training set are passed to the model as **integers** (i.e. 0 or 1) as opposed to one-hot encoded vectors.
  - The model outputs class scores (logits) for the 2 classes.
  - You can browse the [torch.nn loss functions](https://pytorch.org/docs/stable/nn.html#loss-functions) and determine which one is best used for this case.

* For metrics, you can measure `accuracy` inside your training loop.
* For the optimizer, please use [RMSprop](https://pytorch.org/docs/stable/generated/torch.optim.RMSprop.html).
  - Please use a learning rate of `0.001`.
  - Keras' RMSprop defaults are `rho=0.9` and `epsilon=1e-7`; the PyTorch equivalents are the `alpha` and `eps` arguments (PyTorch's own defaults, `0.99` and `1e-8`, take much larger first steps).

In [ ]:
# the labels arrive as integers 0 or 1 and the model outputs 2 logits,
# so cross entropy is the matching loss. It applies the softmax internally.
loss_fn = nn.CrossEntropyLoss()

# RMSprop with a learning rate of 0.001. alpha and eps are set to Keras' defaults
# (rho=0.9, epsilon=1e-7); PyTorch's own defaults take much larger first steps.
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001, alpha=0.9, eps=1e-7)

### Train your model

Please write a training loop that iterates over the training batches and train your model for just **3** epochs.
- **Note:** Please do not exceed 3 epochs so that your results are comparable with the expected output.
  - Afterwards, feel free to continue training to improve your model.

We have loaded pre-trained weights for 15 epochs so you can get a better output when you visualize the saliency maps.

In [ ]:
# load pre-trained weights
load_keras_weights(model, '15_epochs.h5')

# train the model for just 3 epochs
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, seen = 0.0, 0, 0

    for images, labels in train_batches:
        images, labels = images.to(device), labels.to(device)   # shape: (N, 3, 300, 300), (N,)

        optimizer.zero_grad()
        logits = model(images)                                  # shape: (N, 2)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(images)
        correct += (logits.argmax(1) == labels).sum().item()
        seen += len(images)

    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {running_loss / seen:.4f} - accuracy: {correct / seen:.4f}")

### Generate saliency maps at 18 epochs

You will now use your `do_salience()` function again on the same test images. Please use the same parameters as before but this time, use the prefix `salient`.

In [ ]:
# same five images and labels as before, but with the `salient` prefix this time
for image_name, image_label in test_images:
    do_salience(image_name, model, image_label, 'salient')

You should see that the strong pixels are now very less than the ones you generated earlier. Moreover, most of them are now found on features within the pet.

### Zip the images

Please run the cell below to zip the normalized tensor images you generated at 18 epochs. If you get an error, please check that you have files named:

* salientcat1.jpg
* salientcat2.jpg
* salientcatanddog.jpg
* salientdog1.jpg
* salientdog2.jpg

(In the original course this **images.zip** was uploaded to the Coursera grader.)

In [ ]:
from zipfile import ZipFile

if os.path.exists('images.zip'):
    os.remove('images.zip')

filenames = ['cat1.jpg', 'cat2.jpg', 'catanddog.jpg', 'dog1.jpg', 'dog2.jpg']

# writing files to a zipfile
with ZipFile('images.zip','w') as zip_file:
  for file in filenames:
    zip_file.write('salient' + file)

print("images.zip generated!")

### Optional: Saliency Maps at 95 epochs

We have pre-trained weights generated at 95 epochs and you can see the difference between the maps you generated at 18 epochs.

In [ ]:
if not os.path.exists('95_epochs.h5'):
    urllib.request.urlretrieve('https://docs.google.com/uc?export=download&id=14vFpBJsL_TNQeugX8vUTv8dYZxn__fQY', '95_epochs.h5')

load_keras_weights(model, '95_epochs.h5')

do_salience('cat1.jpg', model, 0, "epoch95_salient")
do_salience('cat2.jpg', model, 0, "epoch95_salient")
do_salience('catanddog.jpg', model, 0, "epoch95_salient")
do_salience('dog1.jpg', model, 1, "epoch95_salient")
do_salience('dog2.jpg', model, 1, "epoch95_salient")

**Congratulations on completing this week's assignment!**